In [1]:
# -*- coding: utf-8 -*-
"""
Created on 2026-06-29

@author:       Oscar Trevizo
@institution:  Harvard Extension School — Graduate Data Science Program (2023)
@context:      Independent project — applying course concepts to real-world data
@environment:  Python 3.14.3 | myenv | MacBook Air M5

Agentic Data Product Builder
=============================

Description:
    An agentic AI loop that dynamically composes governed data products from
    available sources. Claude (via the Anthropic API) reasons about which
    sources to load, detects and resolves semantic conflicts, orchestrates the
    pipeline, and returns a DataProduct with full lineage — without any
    hardcoded composition logic.

    Agent tools (8 total):
        list_available_sources()              what data files are on disk
        load_source(name, years)              load and register a DataProduct
        check_semantic_conflicts(s1, s2)      detect name collisions before merge
        rename_semantic_entry(dp, old, new)   resolve a conflict by renaming
        merge_sources(s1, s2, name)           compose two DataProducts into one
        query_product(dp, name, year, top_n)  semantic-layer query on a product
        get_product_card(dp)                  human-readable summary
        get_lineage(dp)                       full audit trail

    Support module  : data_product_lib.py (same folder)
    Agent pattern   : machine_learning/agentic_ai_vignette_yfinance.ipynb

Key Concepts:
    Agentic AI            — LLM reasons and acts through a tool-use loop
    Tool Use              — Claude decides which tools to call and in what order
    Dynamic Composition   — agent builds the data product from a goal, not a script
    Semantic Conflict     — detected and resolved by the agent before merging
    Governed Output       — result is always a DataProduct with lineage + contract

Revision History:
    2026-06-29  Original development
                - 8 tools: list, load, check, rename, merge, query, card, lineage
                - PRODUCT_REGISTRY for stateful multi-step composition
                - Single-question demos and multi-turn chat session
"""

'\nCreated on 2026-06-29\n\n@author:       Oscar Trevizo\n@institution:  Harvard Extension School — Graduate Data Science Program (2023)\n@context:      Independent project — applying course concepts to real-world data\n@environment:  Python 3.14.3 | myenv | MacBook Air M5\n\nAgentic Data Product Builder\n=============================\n\nDescription:\n    An agentic AI loop that dynamically composes governed data products from\n    available sources. Claude (via the Anthropic API) reasons about which\n    sources to load, detects and resolves semantic conflicts, orchestrates the\n    pipeline, and returns a DataProduct with full lineage — without any\n    hardcoded composition logic.\n\n    Agent tools (8 total):\n        list_available_sources()              what data files are on disk\n        load_source(name, years)              load and register a DataProduct\n        check_semantic_conflicts(s1, s2)      detect name collisions before merge\n        rename_semantic_entry(dp, old, 

## How the Agent Works

The same tool-use loop as `agentic_ai_vignette_yfinance.ipynb`, applied to
data product composition instead of stock prices.

```
User goal: "Build a demographics + economics data product"
       ↓
   Claude reasons: I need UN WPP and World Bank GDP
       ↓  calls → list_available_sources()
       ↓  calls → load_source("UN_WPP")
       ↓  calls → load_source("WORLD_BANK_GDP")
       ↓  calls → check_semantic_conflicts("UN_WPP", "WORLD_BANK_GDP")
       ↓  (no conflicts) calls → merge_sources("UN_WPP", "WORLD_BANK_GDP")
       ↓  calls → get_product_card("MERGED")
       ↓
   Claude answers: "Built MERGED — 14,931 rows, 7 semantic entries, 5 lineage steps"
```

The agent's state lives in `PRODUCT_REGISTRY` — a dict that persists across
tool calls within a session. Once a source is loaded, the agent can query,
merge, and inspect it without reloading.

| Component | Role |
|---|---|
| `PRODUCT_REGISTRY` | Stateful store of loaded DataProducts (keyed by name) |
| Tool functions | Python functions the agent can call |
| Tool schemas | JSON descriptions that tell Claude what each tool does |
| Agent loop | send → tool call → execute → feed result back → repeat |

In [2]:
import io
import json
import os
import sys
from datetime import datetime, timezone

import anthropic
import numpy as np
import pandas as pd

from data_product_lib import (
    DataProductMetadata,
    SemanticLayer,
    LineageTracker,
    DataProduct,
)

client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
print(f"Anthropic client ready. SDK: {anthropic.__version__}")

# ── Model selector ───────────────────────────────────────────────────────────
# Change AGENT_MODEL to switch the agent's reasoning engine without touching
# run_agent() or chat_session(). Haiku is fast/cheap; Sonnet reasons better
# on multi-step cross-metric questions.
# AGENT_MODEL = "claude-haiku-4-5-20251001"   # fast, lower cost
AGENT_MODEL = "claude-sonnet-4-6"          # stronger reasoning

Anthropic client ready. SDK: 0.88.0


## Example Questions to Ask the Agent

Ask any of these in the demo cells or in the multi-turn chat at the bottom.

### Composition
- `"What data sources are available?"`
- `"Build me a data product combining demographics and economics for 2000–2019"`
- `"What would I need to add to also track CO₂ emissions per country?"`

### Data queries
- `"Which countries had the highest net migration rate in 2020 and how did their GDP growth compare?"`
- `"Are there countries where fertility rate is rising and GDP is also growing?"`
- `"Which countries have the most complete data across both sources?"`

### Quality and lineage
- `"How fresh is this data product and when should I refresh it?"`
- `"What transformations were applied before this data reached me?"`
- `"Show me the full lineage of the merged product"`

### Semantic layer
- `"Are there any semantic conflicts if I combine UN WPP and World Bank?"`
- `"What business names are available in the merged product?"`
- `"Resolve any conflicts and merge the two sources"`

### Governance
- `"Is this data product safe to share externally given its license?"`

### Planning
- `"What's the minimum set of sources I need to answer questions about migration and economic development?"`
- `"If I wanted to replicate a Harvard statistics project on migration, what sources and steps would I need?"`

In [3]:
# Pipeline functions — lifted from un_wpp_data_product.ipynb and
# un_wpp_wb_data_product.ipynb; reproduced here so the notebook is
# self-contained. The agent calls these indirectly through the tool layer.

UN_FILE          = '../../data/WPP2024_GEN_F01_DEMOGRAPHIC_INDICATORS_COMPACT_20260327.xlsx'
WB_GDP_FILE      = '../../data/WB_GDP.xls'
WB_GDP_GROWTH_FILE = '../../data/WB_GDP_growth.xls'


def load_un_wpp(filepath, year_start=1961, year_end=2023):
    df = pd.read_excel(
        filepath, sheet_name='Estimates', skiprows=16, thousands=' ',
        usecols=[
            'Region, subregion, country or area *', 'ISO3 Alpha-code',
            'ISO2 Alpha-code', 'Type', 'Year',
            'Total Population, as of 1 July (thousands)',
            'Median Age, as of 1 July (years)',
            'Population Growth Rate (percentage)',
            'Total Fertility Rate (live births per woman)',
            'Life Expectancy at Birth, both sexes (years)',
            'Net Number of Migrants (thousands)',
            'Net Migration Rate (per 1,000 population)',
        ]
    )
    df = df.rename(columns={
        'Region, subregion, country or area *'         : 'Location',
        'ISO3 Alpha-code'                              : 'ISO3',
        'ISO2 Alpha-code'                              : 'ISO2',
        'Type'                                         : 'LocType',
        'Total Population, as of 1 July (thousands)'  : 'Population_Ks',
        'Median Age, as of 1 July (years)'             : 'MedAge',
        'Population Growth Rate (percentage)'          : 'PopulationGrowthRate',
        'Total Fertility Rate (live births per woman)' : 'FertilityRate_births_per_woman',
        'Life Expectancy at Birth, both sexes (years)' : 'LifeExpectancy',
        'Net Number of Migrants (thousands)'           : 'NetMigrants_Ks',
        'Net Migration Rate (per 1,000 population)'    : 'NetMigrationRate_per_Kpop',
    })
    df = df.dropna(subset=['Year'])
    df['Year'] = df['Year'].astype(np.int64)
    df = df[(df['Year'] >= year_start) & (df['Year'] <= year_end)]
    df.loc[df['LocType'] == 'Country/Area', 'LocType'] = 'Country'
    df['ReceivesMigrants']    = (df['NetMigrants_Ks'] > 0).astype(int)
    df['ImmigrantsEmigrants'] = df['NetMigrants_Ks'].apply(
        lambda x: 'Immigrants' if x > 0 else 'Emigrants'
    )
    return df.reset_index(drop=True)


def filter_countries(df):
    return df[df['LocType'] == 'Country'].reset_index(drop=True)


def clean_types(df):
    for col in ['Population_Ks', 'MedAge', 'PopulationGrowthRate',
                'FertilityRate_births_per_woman', 'LifeExpectancy',
                'NetMigrants_Ks', 'NetMigrationRate_per_Kpop']:
        df = df.copy()
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def load_wb_gdp(gdp_file, gdp_growth_file, year_start=1961, year_end=2023):
    def _load_one(filepath, indicator):
        df = pd.read_excel(filepath, skiprows=3, sheet_name='Data')
        df = df.rename(columns={'Country Code': 'ISO3'})
        df = df.drop(columns=['Country Name', 'Indicator Name', 'Indicator Code'])
        df = df.drop(columns=[c for c in df.columns if str(c) == '2025'], errors='ignore')
        df = df.set_index('ISO3').stack().reset_index()
        df.columns = ['ISO3', 'Year', indicator]
        df['Year']    = df['Year'].astype(np.int64)
        df[indicator] = pd.to_numeric(df[indicator], errors='coerce')
        return df
    df = pd.merge(_load_one(gdp_file, 'GDP_USD'),
                  _load_one(gdp_growth_file, 'GDP_growth_pct'),
                  on=['ISO3', 'Year'])
    df = df[(df['Year'] >= year_start) & (df['Year'] <= year_end)]
    return df.reset_index(drop=True)


def _merge_data_products(dp1, dp2, on):
    """Inner join two DataProducts; collision check; inherit lineage."""
    collisions = (set(dp1.semantic_layer.to_dict()) &
                  set(dp2.semantic_layer.to_dict()))
    if collisions:
        raise ValueError(
            f'Semantic name collision(s): {sorted(collisions)}. '
            'Use rename_semantic_entry() to resolve before merging.'
        )
    merged_df = pd.merge(dp1.data, dp2.data, on=on, how='inner')
    merged_sl = SemanticLayer()
    for name, e in dp1.semantic_layer.to_dict().items():
        merged_sl.register(name, e['column'], e['unit'],
                           e['description'], e['source_system'])
    for name, e in dp2.semantic_layer.to_dict().items():
        merged_sl.register(name, e['column'], e['unit'],
                           e['description'], e['source_system'])
    merged_lin = LineageTracker()
    for s in dp1.lineage.to_list():
        merged_lin.log(f"dp1/{s['step']}", s['operation'],
                       s['input_shape'], s['output_shape'], s['notes'])
    for s in dp2.lineage.to_list():
        merged_lin.log(f"dp2/{s['step']}", s['operation'],
                       s['input_shape'], s['output_shape'], s['notes'])
    merged_lin.log('merge', 'inner_join', dp1.data.shape, merged_df.shape,
                   f'keys: {"+".join(on)}  |  DP2 input: {dp2.data.shape}')
    return merged_df, merged_sl, merged_lin


print("Pipeline functions ready.")

Pipeline functions ready.


In [4]:
# ── Product registry ────────────────────────────────────────────────────────
# Persists loaded DataProducts across cells within the Jupyter session.
PRODUCT_REGISTRY: dict = {}

SOURCE_CONFIGS = {
    "UN_WPP": {
        "description": "UN World Population Prospects 2024 — demographics",
        "indicators":  ["net_migration_rate", "net_migrants", "population",
                        "fertility_rate", "life_expectancy"],
        "years":       "1961–2023",  "grain": "country-year (237 countries)",
    },
    "WORLD_BANK_GDP": {
        "description": "World Bank WDI — GDP indicators",
        "indicators":  ["gdp", "gdp_growth"],
        "years":       "1961–2023",  "grain": "country-year (266 ISO3 codes)",
    },
}


# ── Tool 1 ──────────────────────────────────────────────────────────────────
def list_available_sources() -> dict:
    return {"available_sources": SOURCE_CONFIGS,
            "loaded": list(PRODUCT_REGISTRY.keys())}


# ── Tool 2 ──────────────────────────────────────────────────────────────────
def load_source(source_name: str,
                year_start: int = 1961,
                year_end:   int = 2023) -> dict:
    if source_name in PRODUCT_REGISTRY:
        dp = PRODUCT_REGISTRY[source_name]
        return {"status": "already_loaded", "product_name": source_name,
                "shape": list(dp.data.shape)}

    if source_name == "UN_WPP":
        lin = LineageTracker()
        raw  = load_un_wpp(UN_FILE, year_start, year_end)
        lin.log("1-load",   "load_un_wpp",       (0,0),      raw.shape,
                f"Estimates, skiprows=16, {year_start}–{year_end}")
        ctry = filter_countries(raw)
        lin.log("2-filter", "filter_countries",   raw.shape,  ctry.shape,
                "LocType == Country")
        df   = clean_types(ctry)
        lin.log("3-clean",  "clean_types",         ctry.shape, df.shape,
                "pd.to_numeric on 7 columns")
        sl = SemanticLayer()
        sl.register("net_migration_rate","NetMigrationRate_per_Kpop",
                    "per 1,000 population","Net migrants per 1,000 residents",
                    source_system="UN WPP 2024")
        sl.register("net_migrants","NetMigrants_Ks","thousands",
                    "Net number of migrants", source_system="UN WPP 2024")
        sl.register("population","Population_Ks","thousands",
                    "Total population, as of 1 July", source_system="UN WPP 2024")
        sl.register("fertility_rate","FertilityRate_births_per_woman",
                    "births per woman","Total fertility rate",
                    source_system="UN WPP 2024")
        sl.register("life_expectancy","LifeExpectancy","years",
                    "Life expectancy at birth", source_system="UN WPP 2024")
        meta = DataProductMetadata(
            name="UN WPP Demographics",
            description=f"Country-year panel, UN WPP 2024, {year_start}–{year_end}",
            domain="Demographics", owner="Agentic Data Product Builder",
            source="UN World Population Prospects 2024",
            source_url="https://population.un.org/wpp/",
            license="CC BY 3.0 IGO", version="1.0",
            refresh_frequency="Every 2 years",
            created_at=datetime.now(timezone.utc).isoformat())
        dp = DataProduct(df, meta, sl, lin)

    elif source_name == "WORLD_BANK_GDP":
        lin = LineageTracker()
        df  = load_wb_gdp(WB_GDP_FILE, WB_GDP_GROWTH_FILE, year_start, year_end)
        lin.log("1-load", "load_wb_gdp", (0,0), df.shape,
                f"GDP_USD + GDP_growth_pct, wide→long, {year_start}–{year_end}")
        sl = SemanticLayer()
        sl.register("gdp","GDP_USD","constant USD",
                    "GDP in constant US dollars (NY.GDP.MKTP.KD)",
                    source_system="World Bank WDI")
        sl.register("gdp_growth","GDP_growth_pct","% per year",
                    "GDP growth rate (NY.GDP.MKTP.KD.ZG)",
                    source_system="World Bank WDI")
        meta = DataProductMetadata(
            name="World Bank GDP Indicators",
            description=f"Country-year panel, World Bank WDI, {year_start}–{year_end}",
            domain="Economics", owner="Agentic Data Product Builder",
            source="World Bank World Development Indicators",
            source_url="https://databank.worldbank.org/",
            license="CC BY 4.0", version="1.0",
            refresh_frequency="Annual",
            created_at=datetime.now(timezone.utc).isoformat())
        dp = DataProduct(df, meta, sl, lin)
    else:
        return {"error": f"Unknown source '{source_name}'.",
                "available": list(SOURCE_CONFIGS.keys())}

    PRODUCT_REGISTRY[source_name] = dp
    return {"status": "loaded", "product_name": source_name,
            "shape":          list(dp.data.shape),
            "semantic_names": list(dp.semantic_layer.to_dict().keys()),
            "lineage_steps":  len(dp.lineage.to_list()),
            "year_range":     f"{year_start}–{year_end}"}


# ── Tool 3 ──────────────────────────────────────────────────────────────────
def check_semantic_conflicts(source1_name: str, source2_name: str) -> dict:
    missing = [n for n in (source1_name, source2_name)
               if n not in PRODUCT_REGISTRY]
    if missing:
        return {"error": f"Not in registry: {missing}. Load them first."}
    n1 = set(PRODUCT_REGISTRY[source1_name].semantic_layer.to_dict())
    n2 = set(PRODUCT_REGISTRY[source2_name].semantic_layer.to_dict())
    cols = sorted(n1 & n2)
    return {"source1": source1_name, "source2": source2_name,
            "collisions": cols, "safe_to_merge": len(cols) == 0,
            "source1_names": sorted(n1), "source2_names": sorted(n2)}


# ── Tool 4 ──────────────────────────────────────────────────────────────────
def rename_semantic_entry(product_name: str,
                           old_name:     str,
                           new_name:     str) -> dict:
    if product_name not in PRODUCT_REGISTRY:
        return {"error": f"'{product_name}' not in registry"}
    sl = PRODUCT_REGISTRY[product_name].semantic_layer
    if old_name not in sl._registry:
        return {"error": f"'{old_name}' not in {product_name}",
                "available": sorted(sl._registry.keys())}
    e = sl._registry[old_name]
    sl.register(new_name, e["column"], e["unit"],
                e["description"], e["source_system"])
    del sl._registry[old_name]
    return {"status": "renamed", "product_name": product_name,
            "old_name": old_name, "new_name": new_name,
            "current_names": sorted(sl._registry.keys())}


# ── Tool 5 ──────────────────────────────────────────────────────────────────
def merge_sources(source1_name: str,
                  source2_name: str,
                  product_name: str = "MERGED") -> dict:
    missing = [n for n in (source1_name, source2_name)
               if n not in PRODUCT_REGISTRY]
    if missing:
        return {"error": f"Not in registry: {missing}. Load them first."}
    dp1, dp2 = PRODUCT_REGISTRY[source1_name], PRODUCT_REGISTRY[source2_name]
    try:
        mdf, msl, mlin = _merge_data_products(dp1, dp2, on=["ISO3","Year"])
    except ValueError as exc:
        return {"error": str(exc),
                "hint": "Use rename_semantic_entry() to resolve, then retry."}
    meta = DataProductMetadata(
        name=product_name,
        description=f"Merged: {source1_name} + {source2_name}",
        domain="Multi-domain", owner="Agentic Data Product Builder",
        source=f"{dp1.metadata.source} + {dp2.metadata.source}",
        source_url=f"{dp1.metadata.source_url} | {dp2.metadata.source_url}",
        license=f"{dp1.metadata.license} + {dp2.metadata.license}",
        version="1.0", refresh_frequency="On demand",
        created_at=datetime.now(timezone.utc).isoformat())
    dp = DataProduct(mdf, meta, msl, mlin)
    PRODUCT_REGISTRY[product_name] = dp
    return {"status": "merged", "product_name": product_name,
            "shape":          list(dp.data.shape),
            "year_range":     f"{int(dp.data.Year.min())}–{int(dp.data.Year.max())}",
            "countries":      int(dp.data.ISO3.nunique()),
            "lineage_steps":  len(dp.lineage.to_list()),
            "semantic_names": sorted(dp.semantic_layer.to_dict().keys())}


# ── Tool 6 ──────────────────────────────────────────────────────────────────
def query_product(product_name:  str,
                  business_name: str,
                  year:          int = None,
                  top_n:         int = 10) -> dict:
    if product_name not in PRODUCT_REGISTRY:
        return {"error": f"'{product_name}' not in registry"}
    dp  = PRODUCT_REGISTRY[product_name]
    sl  = dp.semantic_layer.to_dict()
    if business_name not in sl:
        return {"error": f"'{business_name}' not in semantic layer",
                "available": sorted(sl.keys())}
    entry = sl[business_name]
    col   = entry["column"]
    cols  = [c for c in ["Location","ISO3","Year",col] if c in dp.data.columns]
    df    = dp.data[cols].copy()
    if year is not None:
        df = df[df["Year"] == year]
    result = df.nlargest(top_n, col).reset_index(drop=True)
    return {"business_name":   business_name,
            "resolved_column": col,
            "unit":            entry["unit"],
            "source_system":   entry["source_system"],
            "year_filter":     year, "top_n": top_n,
            "results":         result.to_dict(orient="records")}


# ── Tool 7 ──────────────────────────────────────────────────────────────────
def get_product_card(product_name: str) -> dict:
    if product_name not in PRODUCT_REGISTRY:
        return {"error": f"'{product_name}' not in registry"}
    buf, old = io.StringIO(), sys.stdout
    sys.stdout = buf
    PRODUCT_REGISTRY[product_name].card()
    sys.stdout = old
    return {"product_name": product_name, "card": buf.getvalue()}


# ── Tool 8 ──────────────────────────────────────────────────────────────────
def get_lineage(product_name: str) -> dict:
    if product_name not in PRODUCT_REGISTRY:
        return {"error": f"'{product_name}' not in registry"}
    dp = PRODUCT_REGISTRY[product_name]
    return {"product_name": product_name,
            "step_count":   len(dp.lineage.to_list()),
            "steps":        dp.lineage.to_list()}


print(f"8 tools defined. PRODUCT_REGISTRY is empty — ask the agent to fill it.")

8 tools defined. PRODUCT_REGISTRY is empty — ask the agent to fill it.


In [5]:
TOOLS = [
    {
        "name": "list_available_sources",
        "description": (
            "Return the catalogue of data sources available to load, and which "
            "are already loaded in the registry. Call this first when the user asks "
            "what data is available or before deciding which sources to load."
        ),
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "load_source",
        "description": (
            "Load a named data source and register it as a DataProduct. "
            "Available source names: 'UN_WPP', 'WORLD_BANK_GDP'. "
            "Optionally restrict the year range. Returns shape, semantic names, "
            "and lineage step count."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "source_name": {
                    "type": "string",
                    "description": "Name of the source to load: 'UN_WPP' or 'WORLD_BANK_GDP'"
                },
                "year_start": {
                    "type": "integer",
                    "description": "First year to include, inclusive. Default 1961."
                },
                "year_end": {
                    "type": "integer",
                    "description": "Last year to include, inclusive. Default 2023."
                }
            },
            "required": ["source_name"]
        }
    },
    {
        "name": "check_semantic_conflicts",
        "description": (
            "Check for business-name collisions between two registered DataProducts "
            "before merging. Returns the list of conflicting names and whether it "
            "is safe to merge. Always call this before merge_sources()."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "source1_name": {"type": "string",
                                 "description": "Name of the first registered product"},
                "source2_name": {"type": "string",
                                 "description": "Name of the second registered product"}
            },
            "required": ["source1_name", "source2_name"]
        }
    },
    {
        "name": "rename_semantic_entry",
        "description": (
            "Rename a business name in a registered DataProduct's semantic layer "
            "to resolve a naming conflict before merging. For example, rename "
            "'population' in one source to 'wpp_population' so it no longer "
            "collides with the other source's 'population'."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "product_name": {"type": "string",
                                 "description": "Name of the registered product to modify"},
                "old_name": {"type": "string",
                             "description": "Business name to rename"},
                "new_name": {"type": "string",
                             "description": "New business name to assign"}
            },
            "required": ["product_name", "old_name", "new_name"]
        }
    },
    {
        "name": "merge_sources",
        "description": (
            "Inner join two registered DataProducts on ISO3 + Year. "
            "Inherits semantic layers and lineage from both parents. "
            "Returns error if semantic conflicts exist — resolve them first "
            "with rename_semantic_entry(). The merged product is registered "
            "under product_name (default 'MERGED')."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "source1_name": {"type": "string",
                                 "description": "Name of the first (left/primary) product"},
                "source2_name": {"type": "string",
                                 "description": "Name of the second product"},
                "product_name": {"type": "string",
                                 "description": "Registry name for the merged result. Default 'MERGED'."}
            },
            "required": ["source1_name", "source2_name"]
        }
    },
    {
        "name": "query_product",
        "description": (
            "Query a registered data product by semantic business name. "
            "Returns the top-N rows by that metric, optionally filtered to a "
            "specific year. Use this to answer questions like 'which countries "
            "had the highest net migration rate in 2020'."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "product_name":  {"type": "string",
                                  "description": "Name of the registered product to query"},
                "business_name": {"type": "string",
                                  "description": "Semantic business name, e.g. 'net_migration_rate', 'gdp_growth'"},
                "year":          {"type": "integer",
                                  "description": "Filter to a specific year. Omit for all years."},
                "top_n":         {"type": "integer",
                                  "description": "Number of top rows to return. Default 10."}
            },
            "required": ["product_name", "business_name"]
        }
    },
    {
        "name": "get_product_card",
        "description": (
            "Return the human-readable card summary of a registered data product: "
            "metadata, semantic layer, quality, and lineage. Use this to give the "
            "user a full overview of what was built."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "product_name": {"type": "string",
                                 "description": "Name of the registered product"}
            },
            "required": ["product_name"]
        }
    },
    {
        "name": "get_lineage",
        "description": (
            "Return the full lineage audit trail of a registered data product: "
            "every transformation step, its input/output shape, and timestamp. "
            "Use this when the user asks about provenance or what was done to the data."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "product_name": {"type": "string",
                                 "description": "Name of the registered product"}
            },
            "required": ["product_name"]
        }
    }
]

TOOL_FUNCTIONS = {
    "list_available_sources":  list_available_sources,
    "load_source":             load_source,
    "check_semantic_conflicts": check_semantic_conflicts,
    "rename_semantic_entry":   rename_semantic_entry,
    "merge_sources":           merge_sources,
    "query_product":           query_product,
    "get_product_card":        get_product_card,
    "get_lineage":             get_lineage,
}

print(f"{len(TOOLS)} tool schemas registered with Claude.")

8 tool schemas registered with Claude.


In [6]:
SYSTEM_PROMPT = """
You are a Data Product Engineer agent. You compose governed, self-describing
data products from available sources using the tools provided.

When given a request:
1. Call list_available_sources() if you are unsure what is on disk.
2. Call load_source() for each source you need.
3. Call check_semantic_conflicts() before any merge.
4. If conflicts exist, call rename_semantic_entry() to resolve them, then retry.
5. Call merge_sources() to compose the final product.
6. Answer the user's question with query_product() or get_product_card().

Always include in your response:
- Shape (rows × columns) of the final product
- Year range and country count
- Number of lineage steps
- Semantic names available

When resolving semantic conflicts, explain your renaming choice briefly
(e.g. "I renamed UN WPP's 'population' to 'wpp_population' to avoid
shadowing World Bank's 'population' entry.").
Be concise and data-driven.
"""


def run_agent(user_question: str, verbose: bool = True) -> str:
    """
    Run the data product agent on a single question.

    The agent loop:
      1. Send question to Claude with all 8 tool schemas
      2. If Claude calls a tool: execute the Python function, feed result back
      3. When Claude returns text (stop_reason='end_turn'): done

    Parameters
    ----------
    user_question : str   Natural-language question or goal.
    verbose       : bool  Print each tool call as it happens (default True).

    Returns
    -------
    str  Claude's final response.
    """
    messages      = [{"role": "user", "content": user_question}]
    max_iters     = 20   # safety guard — complex compositions need more steps

    for _ in range(max_iters):

        response = client.messages.create(
            model      = AGENT_MODEL,
            max_tokens = 2048,
            system     = SYSTEM_PROMPT,
            tools      = TOOLS,
            messages   = messages,
        )

        if response.stop_reason == "end_turn":
            for block in response.content:
                if hasattr(block, "text"):
                    return block.text

        elif response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        print(f"  → {block.name}({block.input})")
                    fn     = TOOL_FUNCTIONS.get(block.name)
                    result = fn(**block.input) if fn else {"error": "unknown tool"}
                    tool_results.append({
                        "type":        "tool_result",
                        "tool_use_id": block.id,
                        "content":     json.dumps(result),
                    })
            messages.append({"role": "user", "content": tool_results})

    return "Max iterations reached — the agent did not finish."


print("Agent loop ready. Call run_agent('your question') to start.")

Agent loop ready. Call run_agent('your question') to start.


---
## Demo 1 — Explore Available Sources

Ask the agent what data is on disk. No sources are loaded yet.

In [7]:
PRODUCT_REGISTRY.clear()   # fresh start

q = "What data sources are available, and what indicators does each one have?"
print(f"Q: {q}")
print("-" * 60)
print(run_agent(q))

Q: What data sources are available, and what indicators does each one have?
------------------------------------------------------------
  → list_available_sources({})
Here's a summary of what's available:

---

## 📦 Available Data Sources

### 1. 🌍 UN World Population Prospects (UN_WPP)
- **Description:** UN WPP 2024 — demographic data
- **Coverage:** 1961–2023 | 237 countries (country-year grain)
- **Indicators (5):**
  | Business Name | Description |
  |---|---|
  | `net_migration_rate` | Net migration rate per population |
  | `net_migrants` | Absolute number of net migrants |
  | `population` | Total population |
  | `fertility_rate` | Total fertility rate |
  | `life_expectancy` | Life expectancy at birth |

---

### 2. 💰 World Bank WDI — GDP (WORLD_BANK_GDP)
- **Description:** World Bank World Development Indicators — GDP data
- **Coverage:** 1961–2023 | 266 ISO3 codes (country-year grain)
- **Indicators (2):**
  | Business Name | Description |
  |---|---|
  | `gdp` | Gross Dome

---
## Demo 2 — Build a Data Product from a Goal

The agent loads both sources, checks for conflicts, merges, and reports
the governed result — all from a single natural-language instruction.

In [8]:
PRODUCT_REGISTRY.clear()

q = ("Build me a data product combining UN demographics and World Bank GDP "
     "for the years 2000 to 2019. Show me the card when done.")
print(f"Q: {q}")
print("-" * 60)
print(run_agent(q))

Q: Build me a data product combining UN demographics and World Bank GDP for the years 2000 to 2019. Show me the card when done.
------------------------------------------------------------
  → load_source({'source_name': 'UN_WPP', 'year_start': 2000, 'year_end': 2019})
  → load_source({'source_name': 'WORLD_BANK_GDP', 'year_start': 2000, 'year_end': 2019})
  → check_semantic_conflicts({'source1_name': 'UN_WPP', 'source2_name': 'WORLD_BANK_GDP'})
  → merge_sources({'source1_name': 'UN_WPP', 'source2_name': 'WORLD_BANK_GDP', 'product_name': 'UN_WPP_WB_GDP_2000_2019'})
  → get_product_card({'product_name': 'UN_WPP_WB_GDP_2000_2019'})
Here's a full summary of your newly built data product:

---

## 📦 Data Product: `UN_WPP_WB_GDP_2000_2019`

| Attribute | Value |
|---|---|
| **Shape** | 4,107 rows × 16 columns |
| **Year Range** | 2000 – 2019 |
| **Countries** | 210 |
| **Lineage Steps** | 5 |
| **Sources** | UN World Population Prospects 2024 + World Bank WDI |
| **Licenses** | CC BY 3.0 I

---
## Demo 3 — Cross-Source Query

Sources are already loaded from Demo 2. The agent queries the merged
product using business names from two different sources.

In [9]:
# Registry already populated from Demo 2 — no reload needed

q = ("Which 10 countries had the highest net migration rate in 2015, "
     "and what was their GDP growth that year?")
print(f"Q: {q}")
print("-" * 60)
print(run_agent(q))

Q: Which 10 countries had the highest net migration rate in 2015, and what was their GDP growth that year?
------------------------------------------------------------
  → load_source({'source_name': 'UN_WPP'})
  → load_source({'source_name': 'WORLD_BANK_GDP'})
  → check_semantic_conflicts({'source1_name': 'UN_WPP', 'source2_name': 'WORLD_BANK_GDP'})
  → merge_sources({'source1_name': 'UN_WPP', 'source2_name': 'WORLD_BANK_GDP', 'product_name': 'WPP_GDP_MERGED'})
  → query_product({'product_name': 'WPP_GDP_MERGED', 'business_name': 'net_migration_rate', 'top_n': 10, 'year': 2015})
  → query_product({'product_name': 'WPP_GDP_MERGED', 'business_name': 'gdp_growth', 'top_n': 10, 'year': 2015})
Here's the full picture for **2015** — the 10 countries with the highest net migration rate, paired with their GDP growth:

---

### 🌍 Top 10 Countries by Net Migration Rate in 2015

| # | Country | Net Migration Rate *(per 1,000 pop)* | GDP Growth *(% per year)* |
|---|---------|--------------------

---
## Demo 4 — Semantic Conflict Detection and Resolution

Manually inject a conflicting business name into the World Bank product,
then ask the agent to merge — it will detect the conflict, rename to resolve
it, and proceed.

In [10]:
# Ensure both sources are loaded
if "UN_WPP" not in PRODUCT_REGISTRY:
    load_source("UN_WPP", 2000, 2019)
if "WORLD_BANK_GDP" not in PRODUCT_REGISTRY:
    load_source("WORLD_BANK_GDP", 2000, 2019)

# Inject a naming conflict: register 'population' in the WB semantic layer
# (World Bank GDP_USD could plausibly be labelled 'population' by a different
# team — this simulates that scenario)
PRODUCT_REGISTRY["WORLD_BANK_GDP"].semantic_layer.register(
    "population", "GDP_USD", "constant USD",
    "Mislabelled entry — simulates a naming conflict with UN WPP",
    source_system="World Bank WDI"
)
print("Conflict injected: 'population' now exists in both UN_WPP and WORLD_BANK_GDP.")
print()

q = ("Merge UN_WPP and WORLD_BANK_GDP into a product called CONFLICT_TEST. "
     "Handle any semantic conflicts you find.")
print(f"Q: {q}")
print("-" * 60)
print(run_agent(q))

Conflict injected: 'population' now exists in both UN_WPP and WORLD_BANK_GDP.

Q: Merge UN_WPP and WORLD_BANK_GDP into a product called CONFLICT_TEST. Handle any semantic conflicts you find.
------------------------------------------------------------
  → load_source({'source_name': 'UN_WPP'})
  → load_source({'source_name': 'WORLD_BANK_GDP'})
  → check_semantic_conflicts({'source1_name': 'UN_WPP', 'source2_name': 'WORLD_BANK_GDP'})
  → rename_semantic_entry({'product_name': 'UN_WPP', 'old_name': 'population', 'new_name': 'wpp_population'})
  → check_semantic_conflicts({'source1_name': 'UN_WPP', 'source2_name': 'WORLD_BANK_GDP'})
  → merge_sources({'source1_name': 'UN_WPP', 'source2_name': 'WORLD_BANK_GDP', 'product_name': 'CONFLICT_TEST'})
  → get_product_card({'product_name': 'CONFLICT_TEST'})
Here's a full summary of the **CONFLICT_TEST** data product:

---

## ✅ CONFLICT_TEST — Data Product Summary

### 🔀 Conflict Resolution
One semantic collision was detected and resolved:
- **`po

---
## Multi-Turn Chat Session

The agent remembers context across turns. You can load sources in one
message and query them in the next without repeating yourself.

Try this sequence:
1. `"Load UN WPP and World Bank GDP for 2000–2019"`
2. `"Merge them"`
3. `"Which countries had the highest fertility rate in 2010?"`
4. `"Now show me the lineage"`
5. `"Is this data product safe to share externally?"`

Type `quit` to exit.

In [11]:
def chat_session():
    """
    Multi-turn chat with the data product agent.
    Conversation history is maintained across turns within the session.
    PRODUCT_REGISTRY state persists too — load once, query many times.
    """
    PRODUCT_REGISTRY.clear()
    messages = []
    print("Data Product Agent — multi-turn chat. Type 'quit' to exit.")
    print("=" * 60)

    while True:
        user_input = input("\nYou: ").strip()
        if user_input.lower() in ("quit", "exit", "q"):
            print("Session ended.")
            break
        if not user_input:
            continue

        messages.append({"role": "user", "content": user_input})

        for _ in range(20):
            response = client.messages.create(
                model      = AGENT_MODEL,
                max_tokens = 2048,
                system     = SYSTEM_PROMPT,
                tools      = TOOLS,
                messages   = messages,
            )
            if response.stop_reason == "end_turn":
                for block in response.content:
                    if hasattr(block, "text"):
                        print(f"\nAgent: {block.text}")
                        messages.append({"role": "assistant",
                                         "content": response.content})
                break
            elif response.stop_reason == "tool_use":
                messages.append({"role": "assistant", "content": response.content})
                tool_results = []
                for block in response.content:
                    if block.type == "tool_use":
                        print(f"  → {block.name}({block.input})")
                        fn     = TOOL_FUNCTIONS.get(block.name)
                        result = fn(**block.input) if fn else {"error": "unknown tool"}
                        tool_results.append({
                            "type":        "tool_result",
                            "tool_use_id": block.id,
                            "content":     json.dumps(result),
                        })
                messages.append({"role": "user", "content": tool_results})


chat_session()

Data Product Agent — multi-turn chat. Type 'quit' to exit.



You:  Which country in the top 20 GDP has the highest migration rate


  → list_available_sources({})
  → load_source({'source_name': 'UN_WPP'})
  → load_source({'source_name': 'WORLD_BANK_GDP'})
  → check_semantic_conflicts({'source1_name': 'UN_WPP', 'source2_name': 'WORLD_BANK_GDP'})
  → merge_sources({'source1_name': 'UN_WPP', 'source2_name': 'WORLD_BANK_GDP', 'product_name': 'MERGED'})
  → query_product({'product_name': 'MERGED', 'business_name': 'gdp', 'top_n': 20, 'year': 2023})
  → query_product({'product_name': 'MERGED', 'business_name': 'net_migration_rate', 'top_n': 20, 'year': 2023})

Agent: I have everything I need. Here's the full analysis:

---

## 🌍 Highest Migration Rate Among Top-20 GDP Countries (2023)

By cross-referencing the **Top 20 GDP countries** with their **net migration rates**, two countries from the GDP top-20 appear in the global migration rate top-20:

| Rank (by GDP) | Country | GDP (USD) | Net Migration Rate (per 1,000 pop) |
|---|---|---|---|
| 🥇 **#11** | **🇨🇦 Canada** | $1.81 Trillion | **+11.04** |
| #19 | 🇸🇦 Saudi Ara


You:  Load the top 20 countries by GDP in 2023. Then, for just those 20 countries, tell me which has the highest net migration rate.


  → query_product({'product_name': 'MERGED', 'business_name': 'gdp', 'top_n': 20, 'year': 2023})
  → query_product({'product_name': 'MERGED', 'business_name': 'net_migration_rate', 'top_n': 210, 'year': 2023})

Agent: I now have everything needed. Let me cross-reference the two lists precisely.

Here is the definitive ranking of the **Top 20 GDP countries by their net migration rate in 2023**, filtered strictly to only those 20 nations:

---

## 🏆 Net Migration Rate — Ranked Within the Top 20 GDP Countries (2023)

| Rank | Country | GDP (USD) | Net Migration Rate (per 1,000 pop) |
|------|---------|-----------|----------------------------------|
| 🥇 1 | 🇨🇦 **Canada** | $1.81T | **+11.04** |
| 2 | 🇸🇦 Saudi Arabia | $0.86T | +9.79 |
| 3 | 🇳🇱 Netherlands | $0.91T | +7.99 |
| 4 | 🇩🇪 Germany | $3.70T | +7.21 |
| 5 | 🇬🇧 United Kingdom | $3.28T | +6.49 |
| 6 | 🇦🇺 Australia | $1.65T | +5.30 |
| 7 | 🇨🇭 Switzerland | $0.80T | +5.08 |
| 8 | 🇺🇸 United States | $21.96T | +3.85 |
| 9 | 🇮🇹 Italy | $2


You:  Load the top 20 countries by GDP in 2023. Then, for just those 20 countries, tell me which has the highest net migration rate. In the answer, produce a table with Rank (migration rate), Country, GDP (USD), GDP (Ranking in the world), Net Migration rate (per 1,000 pop)


  → query_product({'product_name': 'MERGED', 'business_name': 'gdp', 'top_n': 20, 'year': 2023})
  → query_product({'product_name': 'MERGED', 'business_name': 'net_migration_rate', 'top_n': 210, 'year': 2023})

Agent: Got everything. Here is the precise cross-referenced table, filtered to only the top-20 GDP countries, ranked by net migration rate (highest to lowest):

---

## 🌍 Top 20 GDP Countries — Ranked by Net Migration Rate (2023)

| Migration Rank | Country | GDP (USD) | GDP World Rank | Net Migration Rate (per 1,000 pop) |
|:-:|---|---:|:-:|---:|
| 🥇 1 | 🇨🇦 Canada | $1.81T | #11 | **+11.04** |
| 2 | 🇸🇦 Saudi Arabia | $0.86T | #19 | +9.79 |
| 3 | 🇳🇱 Netherlands | $0.91T | #18 | +7.99 |
| 4 | 🇩🇪 Germany | $3.70T | #4 | +7.21 |
| 5 | 🇬🇧 United Kingdom | $3.28T | #5 | +6.49 |
| 6 | 🇦🇺 Australia | $1.65T | #12 | +5.30 |
| 7 | 🇨🇭 Switzerland | $0.80T | #20 | +5.08 |
| 8 | 🇺🇸 United States | $21.96T | #1 | +3.85 |
| 9 | 🇮🇹 Italy | $2.02T | #8 | +2.52 |
| 10 | 🇪🇸 Spain | $1.38T | #14 |

KeyboardInterrupt: Interrupted by user

---
## Observations & Takeaways

These notes capture findings from live runs of this vignette. They are not
hypothetical — each observation comes from an actual agent session.

---

### 1. The agent reasons differently on the same question

Running the same question twice does not guarantee the same answer, even with
the same model. The tool call sequence — which tools the agent calls, in what
order, and with what parameters — varies between runs. The final answer may
agree while the reasoning path differs.

**Example:** "Which country in the top 20 GDP has the highest migration rate?"

| Run | Strategy | Result |
|-----|----------|--------|
| Haiku (run 1) | Reasoned from a single query — returned USA incorrectly | ❌ Wrong |
| Haiku (run 2) | Queried GDP top 20, then migration top 100, cross-referenced | ✅ Canada |
| Sonnet | Queried GDP top 20, then migration top 20, intersected both lists | ✅ Canada (different scope) |

The answer (Canada) was consistent across correct runs. The *path* to it was not.

---

### 2. Models answer the question they infer, not always the one you meant

Sonnet and Haiku both correctly identified Canada — but answered subtly different
questions.

**What was intended:**
> Filter to the 20 largest economies. Within that set, rank by migration rate.
> Return the winner.

**What Haiku did:** Ranked all top-20 GDP countries by migration rate → produced
a full table (Canada, Netherlands, Germany, UK, Australia, Switzerland, ...).

**What Sonnet did:** Found countries that are simultaneously in the global
top-20 by GDP *and* the global top-20 by migration rate → produced only Canada
and Saudi Arabia.

Sonnet's interpretation is mathematically stricter. Haiku's is closer to the
user's intent. Neither model "read the mind" of the questioner — each made a
plausible inference about scope.

**Practical implication:** For cross-metric questions, spelling out the two
steps explicitly removes the ambiguity:
> *"Get the top 20 countries by GDP. For just those 20 countries, which has
> the highest net migration rate?"*

**Confirmed in session.** The same model (Sonnet) produced different tool
call strategies depending solely on how the question was phrased:

| Question phrasing | Migration query | Coverage | Output |
|---|---|---|---|
| Vague ("top 20 GDP... highest migration") | `top_n=20` | Global top-20 intersection | Canada + Saudi Arabia only |
| Explicit two-step | `top_n=210` | All countries → filtered to GDP top-20 | Full ranked table of all 20 |

---

### 3. Tool call traces are the ground truth

Every agent run prints its tool calls:
```
→ list_available_sources({})
→ load_source({'source_name': 'UN_WPP'})
→ query_product({'business_name': 'gdp', 'top_n': 20, 'year': 2023})
→ query_product({'business_name': 'net_migration_rate', 'top_n': 100, 'year': 2023})
```

These traces tell you exactly what the agent computed vs. what it inferred.
If a result looks surprising, read the trace first — the answer is usually there.
A `top_n=20` on migration rate produces a different answer than `top_n=100`
because the intersection with the GDP list changes.

---

### 4. Agent reliability depends on how cleanly a question maps to tools

The agent is reliable when one tool call answers the question:
> "What sources are available?" → `list_available_sources()` → done.

It introduces variance when the answer requires reasoning across multiple tool
outputs (filtering, ranking, intersecting). That gap between tool capability
and question intent is where models diverge.

| Question type | Reliability | Why |
|---|---|---|
| Single-source lookup | High | One tool call, deterministic result |
| Cross-metric ranking | Medium | Agent must reason across two queries |
| Conflict detection + resolution | High | Explicit tool for each step |
| Lineage / card / schema | High | One tool call, deterministic result |

---

### 5. The PRODUCT_REGISTRY behaves like a data catalog

Once a source is loaded, any subsequent tool call finds it in the registry
without re-reading the file. Within a chat session, this means:
- Loading UN WPP (a large Excel file) happens once per kernel session
- Follow-up questions reuse the registered DataProduct instantly
- The agent learns "this source is already loaded" from the `already_loaded`
  status and skips the load step

This mirrors how a production data catalog works: publish once, consume many
times, with governance metadata attached at registration.

---

### 6. Model selection is a reasoning parameter, not just a cost parameter

Switching from Haiku to Sonnet changed the agent's behavior in observable ways:

| Dimension | Haiku | Sonnet |
|---|---|---|
| Query scope | Wider net (top_n=100) | Prompt-dependent: top_n=20 (vague question) or top_n=210 (explicit two-step) |
| Output style | Plain text table | Formatted with icons and headers |
| Saudi Arabia surfaced? | No (buried in wider list) | Yes (appeared in top-20 intersection) |
| Consistency across runs | Varied | More consistent |

The model is not a fixed component of the pipeline — it is a parameter that
affects reasoning style, not just speed and cost. Choosing a model is choosing
a reasoning trade-off.

---
## References

### Data Sources
- **UN World Population Prospects 2024** — United Nations Department of Economic
  and Social Affairs, Population Division.
  Demographic Indicators (Estimates, 1950–2023).
  License: CC BY 3.0 IGO.
  [https://population.un.org/wpp/](https://population.un.org/wpp/)

- **World Bank World Development Indicators** — GDP (NY.GDP.MKTP.KD) and
  GDP growth (NY.GDP.MKTP.KD.ZG), constant USD.
  License: CC BY 4.0.
  [https://databank.worldbank.org/](https://databank.worldbank.org/)

### Data Product Concepts
- Dehghani, Z. (2022). *Data Mesh: Delivering Data-Driven Value at Scale.*
  O'Reilly Media. — Origin of the data product paradigm; discoverable,
  self-describing, governed, interoperable data units.

- Fowler, M. *Data Mesh Principles and Logical Architecture.*
  [martinfowler.com/articles/data-mesh-principles.html](https://martinfowler.com/articles/data-mesh-principles.html)

- *Data Contract Specification* — datacontract.com. Inspiration for the
  `.contract()` export format.

- *OpenLineage* — openlineage.io. Conceptual basis for the `LineageTracker`
  append-only log pattern.

- *dbt Semantic Layer / Looker LookML* — Conceptual basis for the
  `SemanticLayer` business-name registry pattern.

### Agentic AI
- *Anthropic Tool Use Documentation* — How Claude handles `tool_use` /
  `tool_result` message turns.
  [docs.anthropic.com/en/docs/tool-use](https://docs.anthropic.com/en/docs/tool-use)

### Local Module
- `data_product_lib.py` — Custom module in this folder (not a published
  package). Four classes: `DataProductMetadata`, `SemanticLayer`,
  `LineageTracker`, `DataProduct`. See `data_product_lib_tutorial.md`
  for full API reference and origin notes.